<a href="https://colab.research.google.com/github/zeeshansafdar752/Research_Paper/blob/main/ResearchPaper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Import Libraries

2 Load Dataset

3 Check Dataset

4 Clean Data

5 Define Target

6 Train / Validation / Test Split

7 Preprocessing

8 Baseline Model (Random Forest)

9 Choose Algorithm

10 Define Loss, Activation Function aur Optimizer

11 Training Loop (validation, early stopping)

12 Check Training Output (learning curves)

13 Model Evaluation (test set par, confusion matrix)

14 Error Analysis

15 Repeat with Multiple Seeds

16 Save Model


**# Import Libraries**

In [20]:
# Import Libraries
import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import zipfile
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, confusion_matrix, roc_auc_score)

**Select Device CPU or GPU**

In [21]:
SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


**# Drive mount**

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import zipfile
ZIP_PATH = "/content/drive/MyDrive/Colab Notebooks/xfl/detection+of+iot+botnet+attacks+n+baiot (1).zip"

with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    print("total files:", len(names))
    for n in names[:15]:
        print(n)

total files: 36
Danmini_Doorbell/
Danmini_Doorbell/benign_traffic.csv
Danmini_Doorbell/mirai_attacks.rar
Danmini_Doorbell/gafgyt_attacks.rar
demonstrate_structure.csv
Ecobee_Thermostat/
Ecobee_Thermostat/benign_traffic.csv
Ecobee_Thermostat/mirai_attacks.rar
Ecobee_Thermostat/gafgyt_attacks.rar
Ennio_Doorbell/
Ennio_Doorbell/gafgyt_attacks.rar
Ennio_Doorbell/benign_traffic.csv
N_BaIoT_dataset_description_v1.txt
Philips_B120N10_Baby_Monitor/
Philips_B120N10_Baby_Monitor/benign_traffic.csv


**Load Dataset**

In [25]:
import zipfile
import pandas as pd

# The error happened because 'train_test_network.csv' was not found in your Drive or local directory.
# Based on the previous cell, you are working with the N_BaIoT dataset zip file.
# Let's extract and load one of the CSV files from your ZIP archive instead.

ZIP_PATH = "/content/drive/MyDrive/Colab Notebooks/xfl/detection+of+iot+botnet+attacks+n+baiot (1).zip"

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    # Load the Danmini Doorbell benign traffic as an example
    with z.open('Danmini_Doorbell/benign_traffic.csv') as f:
        df = pd.read_csv(f, low_memory=False)

print("Dataset loaded successfully!")
print(df.shape)
print(df.head())

Dataset loaded successfully!
(49548, 115)
   MI_dir_L5_weight  MI_dir_L5_mean  MI_dir_L5_variance  MI_dir_L3_weight  \
0          1.000000       60.000000            0.000000          1.000000   
1          1.000000      354.000000            0.000000          1.000000   
2          1.857879      360.458980           35.789338          1.912127   
3          1.000000      337.000000            0.000000          1.000000   
4          1.680223      172.140917        18487.448750          1.793580   

   MI_dir_L3_mean  MI_dir_L3_variance  MI_dir_L1_weight  MI_dir_L1_mean  \
0       60.000000            0.000000          1.000000       60.000000   
1      354.000000            0.000000          1.000000      354.000000   
2      360.275733           35.923972          1.969807      360.091968   
3      337.000000            0.000000          1.000000      337.000000   
4      182.560279        18928.175300          1.925828      193.165753   

   MI_dir_L1_variance  MI_dir_L0.1_weight  .

In [26]:
df.head()

,MI_dir_L5_weight,MI_dir_L5_mean,MI_dir_L5_variance,MI_dir_L3_weight,MI_dir_L3_mean,MI_dir_L3_variance,MI_dir_L1_weight,MI_dir_L1_mean,MI_dir_L1_variance,MI_dir_L0.1_weight,...,HpHp_L0.1_radius,HpHp_L0.1_covariance,HpHp_L0.1_pcc,HpHp_L0.01_weight,HpHp_L0.01_mean,HpHp_L0.01_std,HpHp_L0.01_magnitude,HpHp_L0.01_radius,HpHp_L0.01_covariance,HpHp_L0.01_pcc
0,1.000000,60.000000,0.000000,1.000000,60.000000,0.000000,1.000000,60.000000,0.000000,1.000000,...,0.000000,0.0,0.0,1.000000,60.000000,0.000000,60.000000,0.000000,0.0,0.0
1,1.000000,354.000000,0.000000,1.000000,354.000000,0.000000,1.000000,354.000000,0.000000,1.000000,...,34.095047,0.0,0.0,5.319895,344.262695,4.710446,344.262695,22.188299,0.0,0.0
2,1.857879,360.458980,35.789338,1.912127,360.275733,35.923972,1.969807,360.091968,35.991542,1.996939,...,100.081513,0.0,0.0,6.318264,347.703087,9.034660,347.703087,81.625077,0.0,0.0
3,1.000000,337.000000,0.000000,1.000000,337.000000,0.000000,1.000000,337.000000,0.000000,1.000000,...,0.000000,0.0,0.0,1.000000,337.000000,0.000000,337.000000,0.000000,0.0,0.0
4,1.680223,172.140917,18487.448750,1.793580,182.560279,18928.175300,1.925828,193.165753,19153.795810,1.992323,...,0.000000,0.0,0.0,1.000000,60.000000,0.000000,60.000000,0.000000,0.0,0.0


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49548 entries, 0 to 49547
Columns: 115 entries, MI_dir_L5_weight to HpHp_L0.01_pcc
dtypes: float64(115)
memory usage: 43.5 MB


In [29]:
print("missing values:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())

# The loaded CSV file only contains traffic features.
# Since we loaded 'benign_traffic.csv', we can define 'label' as 0 (benign) and 'type' as 'benign'.
if "label" not in df.columns:
    df["label"] = 0
if "type" not in df.columns:
    df["type"] = "benign"

print("\n--- Label Distributions ---")
print(df["type"].value_counts())
print(df["label"].value_counts())

missing values: 0
duplicate rows: 9153

--- Label Distributions ---
type
benign    49548
Name: count, dtype: int64
label
0    49548
Name: count, dtype: int64
